In [ ]:
library(dplyr)
library(sccomp)
library(ggplot2)
library(forcats)
library(tidyr)
library(openxlsx)
library(cmdstanr)
library(ggplot2)
library(cowplot)
library(ggpubr)
library(rstatix)
library(tibble)
library(future)
library(loo)
set_cmdstan_path('/home/liyanguo/software/cmdstan-2.36.0')
plan(multisession)
mc.cores = 48
set.seed(0)


Attaching package: ‘dplyr’


The following objects are masked from ‘package:stats’:

    filter, lag


The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union


Loading required package: instantiate

This is cmdstanr version 0.9.0

- CmdStanR documentation and vignettes: mc-stan.org/cmdstanr

- CmdStan path: /home/liyanguo/software/cmdstan-2.36.0

- CmdStan version: 2.36.0



# PBMC
#Exclude NDN+basophils+platelet

## Classification_L4

### Data loading for sccomp

In [ ]:
group = 'Pregnancy.stage'
classification = 'Classification_L4_PBMC_pregnancy'
#output
path = paste0('/home/liyanguo/MyImmuCell/08_supp_outputs/sccomp_age/',classification,'/')
system(paste0("mkdir -p ",path))
system(paste0("rm ",path,'*'))

In [ ]:
Level_counts_frac = read.xlsx("/home/liyanguo/MyImmuCell/06_Finnal_raw_count/Classification_L4_donor_counts_frac_clinicl_table.xlsx",
                              rowNames=T)

In [4]:
cell_types = c('HSC/MPP#','CILCP#','CLP#','MEP#','MkP#','MC/MCP#',
               
               'Naïve.B.cells#','Early.memory.B.cells#','Non-switched.memory.B.cells#','Switched.Memory.B.cells#',
               'CD95.memory.B.cells#',
               'Atypical.naïve.B.cells#','CD27+.IgD+.atypical.memory.B.cells#','CD27+.IgD-.atypical.memory.B.cells#','CD27-.IgD-.atypical.memory.B.cells#',
               'Plamsablasts#','IGKChi.Plasma.cells#','IGLL5hi.Plasma.cells#',
               
               'DN.T.cells#','Proliferative.DN.T.cells#',
               
               'Naïve.CD4+.T.cells#',
               'Tfh#','CD27+.Th1#','CD27-.Th1#','Th1/Th17#','CD27+.Th17#','CD27-.Th17#','Th2#','Th22#','Proliferative.help.memory.T.cells#',
               'GZMB+.CD4+.terminal.effector.T.cells#','CD4+.Temra#','HLA-DRhi.CD4+.terminal.effector.T.cells#','Proliferative.cytotoxic.CD4+.T.cells#',
               'Naïve.CD4+.Treg#','Memory.CD4+.Treg#','KLRB1+.CD4+.Treg#','HLA-DRhi.CD4+.Treg#','Proliferative.CD4+.Treg#',
               
               'Naïve.CD8+.T.cells#','CCR4+.CD8+.Tcm#','CCR4-.CD8+.Tcm#',
               'GZMK+.CD8+.Tem#','GZMB+.CD8+.Tem#','CD8+.Temra#','HLA-DRhi.CD8+.Tem#','Proliferative.CD8+.memory.T.cells#',
               'CD8+.Treg#',
               
               'CD27+.MAIT#','CD27-.MAIT#','CD56+.MAIT#',
               
               'Naïve.Vδ1+.T.cells#','CD279+.SOX4+.Vδ1+.T.cells#', 'SOX4+.Vδ1+.T.cells#','KLRC2+.effector.Vδ1+.T.cells#','GZMK+.effector.Vδ1+.T.cells#',
               'GZMK+.Vδ2+.T.cells#','GZMB+.Vδ2+.T.cells#','CD62Lhi.GZMK+.Vδ2+.T.cells#','Proliferative.Vδ2+.T.cells#',

               'iNKT#','vNKT#',
               
               'ILCP#','ILC2#','CD56bright.NK.cells#','CD56dim.NK.cells#','Adaptive.NK.cells#','Proliferative.NK.cells#',
               
               'Core.classical.monocytes#','GBP1+.classical.monocytes#','ISG+.classical.monocytes#','Intermediate.monocytes#', 'Non-classical.monocytes#',
               'ASDC#','CD1C+.cDC2#','CD14+.cDC2#','cDC1#','pDCs#','LAMP3+.DC#',
               'MPO+.CD177-.iLDNs#','MPO-.CD177-.iLDNs#','CD177int.iLDNs#','MMP8+.CD177+.iLDNs#','MMP9+.CD177+.iLDNs#','Proliferative.iLDNs#',
               'MPO+.mLDNs#','MMP8+.CD177+.mLDNs#','Proliferative.mLDNs#'
)

In [5]:
#showd be 88 cell type, 排除platelets、NDNs和Bas, 但包含mast cell: 
length(cell_types)

[1] 88

In [6]:
Level_counts_frac_long = Level_counts_frac %>%
    select(ClinicalID,DonorID, Batch, Gender, Age_in_Years,
       `Ten_year_intervals`, Pregnancy, Pregnancy.stage, Sampling.time, all_of(cell_types)) %>%
    pivot_longer(
      cols = -c(ClinicalID,DonorID, Batch, Gender, Age_in_Years, `Ten_year_intervals`, Pregnancy, Pregnancy.stage, Sampling.time),
      names_to = "groupby",
      values_to = "groupby_count"
    ) %>%
    rename(Intervals = `Ten_year_intervals`) %>%
    filter(Pregnancy.stage != 'Parturition') %>% #a single sample
    mutate(
      groupby_count = as.integer(groupby_count),
      Intervals = factor(Intervals, levels = c('2-9','10-19','20-29','30-39','40-49','50-59','60-69','70-79','80-89','90+')),
      Gender = factor(Gender, levels = c('Female','Male')),
      Pregnancy = factor(Pregnancy, levels = c('NO','YES')),
      Sampling.time = factor(Sampling.time, levels = c('Morning','Evening')),
      Pregnancy.stage = factor(Pregnancy.stage, levels = c('Unavailable','First trimester','Second trimester','Third trimester'))
    ) %>% {
        pregnant <- filter(., Pregnancy == 'YES')
        # 计算怀孕样本数量
        n_pregnant <- n_distinct(pregnant$DonorID)
        
        # 非怀孕样本
        non_pregnant <- filter(., Pregnancy == 'NO') %>% filter(., Age_in_Years >= 18 & Age_in_Years <= 40) #%>%
            #distinct(DonorID, .keep_all = TRUE) %>%  # 按样本去重
            #slice_sample(n = n_pregnant, replace = FALSE)  # 无放回抽样
        
        # 合并怀孕样本和对照样本（保留原始长格式结构）
        bind_rows(
            pregnant,
            filter(., DonorID %in% non_pregnant$DonorID)  # 匹配非怀孕样本的所有行
        )
    }

In [7]:
table(Level_counts_frac_long$Pregnancy.stage)


     Unavailable  First trimester Second trimester  Third trimester 
           16720             1584             1056              528 

### Best model and plot

In [8]:
#年龄作为fixed effect
#性别作为fixed effect，但效果一般。已有研究表明性别对于免疫细胞比例影响局限在NK和单核细胞 (Effects of sex and aging on the immune cell landscape as assessed by single-cell transcriptomic analysis)
#仅考虑个体间DonorID的差异和实验批次Batch的差异,使用随机截距Intercept, 旨在捕捉个体和批次基线差异

In [9]:
sccomp_result = Level_counts_frac_long %>%
    sccomp_estimate(
    formula_composition = ~ Pregnancy.stage, # + (1 | DonorID)
    sample = "ClinicalID",
    cell_group = "groupby",
    abundance = "groupby_count",
    bimodal_mean_variability_association = TRUE,
    cores = 48, verbose = FALSE
    ) %>% sccomp_remove_outliers(cores = 48, verbose = FALSE, max_sampling_iterations = 2000) %>% sccomp_test(test_composition_above_logit_fold_change = 0.2)

sccomp says: groupby_count column is an integer. The sum-constrained beta binomial model will be used

sccomp says: estimation

sccomp says: the composition design matrix has columns: (Intercept), Pregnancy.stageFirst trimester, Pregnancy.stageSecond trimester, Pregnancy.stageThird trimester

sccomp says: the variability design matrix has columns: (Intercept)

Loading model from cache...

sccomp says: to do hypothesis testing run `sccomp_test()`,
  the `test_composition_above_logit_fold_change` = 0.1 equates to a change of ~10%, and
  0.7 equates to ~100% increase, if the baseline is ~0.1 proportion.
  Use `sccomp_proportional_fold_change` to convert c_effect (linear) to proportion difference (non-linear).

Loading model from cache...



Running standalone generated quantities after 1 MCMC chain, with 48 thread(s) per chain...

Chain 1 finished in 0.0 seconds.


sccomp says: outlier identification - step 1/2

Loading model from cache...



Running standalone generated quantities after 1 MCMC chain, with 48 thread(s) per chain...

Chain 1 finished in 0.0 seconds.


sccomp says: outlier-free model fitting - step 2/2

sccomp says: the composition design matrix has columns: (Intercept), Pregnancy.stageFirst trimester, Pregnancy.stageSecond trimester, Pregnancy.stageThird trimester

sccomp says: the variability design matrix has columns: (Intercept)

Loading model from cache...



In [10]:
saveRDS(sccomp_result,paste0(dirname(path),'/',classification,"sccomp_result.RDS"))

In [11]:
p1 = sccomp_boxplot(sccomp_result, factor = "Pregnancy.stage",)
ggsave2(paste0(dirname(path),'/',classification, "sccomp_boxplot.pdf"),
        p1,width=80,height=20,limitsize = FALSE)

sccomp says: When visualising proportions, especially for complex models, consider setting `remove_unwanted_effects=TRUE`. This will adjust the proportions, preserving only the observed effect.

Loading model from cache...



Running standalone generated quantities after 1 MCMC chain, with 1 thread(s) per chain...

Chain 1 finished in 0.0 seconds.


Joining with `by = join_by(groupby, ClinicalID)`
Joining with `by = join_by(groupby, Pregnancy.stage)`
Warning message in grid.Call.graphics(C_text, as.graphicsAnnot(x$label), x$x, x$y, :
“conversion failure on 'Naïve.Vδ1+.T.cells#' in 'mbcsToSbcs': for δ (U+03B4)”
Warning message in grid.Call.graphics(C_text, as.graphicsAnnot(x$label), x$x, x$y, :
“conversion failure on 'Proliferative.Vδ2+.T.cells#' in 'mbcsToSbcs': for δ (U+03B4)”
Warning message in grid.Call.graphics(C_text, as.graphicsAnnot(x$label), x$x, x$y, :
“conversion failure on 'SOX4+.Vδ1+.T.cells#' in 'mbcsToSbcs': for δ (U+03B4)”
Warning message in grid.Call.graphics(C_text, as.graphicsAnnot(x$label), x$x, x$y, :
“conversion failure on 'KLRC2+.effector.Vδ1+.T.cells#' in 'mbcsToSbcs': for δ (U+03B4)”
Warning message in grid.Call.graphics(C_text, as.graphicsAnnot(x$label), x$x, x$y, :
“conversion failure on 'CD62Lhi.GZMK+.Vδ2+.T.cells#' in 'mbcsToSbcs': for δ (U+03B4)”
Warning message in grid.Call.graphics(C_text, as.graphic

In [12]:
p2 = plot_1D_intervals(sccomp_result[!grepl("___D", sccomp_result$parameter),])
ggsave2(paste0(dirname(path),'/',classification, "sccomp_plot_1D.pdf"),
        p2,width=20,height=20,limitsize = FALSE)

Warning message in grid.Call.graphics(C_text, as.graphicsAnnot(x$label), x$x, x$y, :
“conversion failure on 'CD279+.SOX4+.Vδ1+.T.cells#' in 'mbcsToSbcs': for δ (U+03B4)”
Warning message in grid.Call.graphics(C_text, as.graphicsAnnot(x$label), x$x, x$y, :
“conversion failure on 'GZMB+.Vδ2+.T.cells#' in 'mbcsToSbcs': for δ (U+03B4)”
Warning message in grid.Call.graphics(C_text, as.graphicsAnnot(x$label), x$x, x$y, :
“conversion failure on 'Proliferative.Vδ2+.T.cells#' in 'mbcsToSbcs': for δ (U+03B4)”
Warning message in grid.Call.graphics(C_text, as.graphicsAnnot(x$label), x$x, x$y, :
“conversion failure on 'GZMK+.effector.Vδ1+.T.cells#' in 'mbcsToSbcs': for δ (U+03B4)”
Warning message in grid.Call.graphics(C_text, as.graphicsAnnot(x$label), x$x, x$y, :
“conversion failure on 'GZMK+.Vδ2+.T.cells#' in 'mbcsToSbcs': for δ (U+03B4)”
Warning message in grid.Call.graphics(C_text, as.graphicsAnnot(x$label), x$x, x$y, :
“conversion failure on 'CD62Lhi.GZMK+.Vδ2+.T.cells#' in 'mbcsToSbcs': for δ 

In [13]:
p3 = plot_2D_intervals(sccomp_result)
ggsave2(paste0(dirname(path),'/',classification, "sccomp_plot_2D.pdf"),
        p3,width=20,height=20,limitsize = FALSE)

In [14]:
head(sccomp_result)

groupby,parameter,factor,c_lower,c_effect,c_upper,c_pH0,c_FDR,c_rhat,c_ess_bulk,c_ess_tail,v_lower,v_effect,v_upper,v_pH0,v_FDR,v_rhat,v_ess_bulk,v_ess_tail
<chr>,<chr>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
ASDC#,(Intercept),NA,-1.9627257,-1.81360442,-1.67875916,0.0000,0.0000000,1.007846,140.5321,348.1280,-10.933447,-10.580325,-10.211768,0,0,1.000573,71.66682,237.9733
ASDC#,Pregnancy.stageFirst trimester,Pregnancy.stage,-0.3000230,-0.13224073,0.03241156,0.7815,0.3067830,1.001602,1598.0329,1827.6465,NA,NA,NA,NA,NA,NA,NA,NA
ASDC#,Pregnancy.stageSecond trimester,Pregnancy.stage,-0.1850229,-0.01181819,0.14602956,0.9830,0.2470966,1.000159,1582.2467,1719.7492,NA,NA,NA,NA,NA,NA,NA,NA
ASDC#,Pregnancy.stageThird trimester,Pregnancy.stage,-0.2897007,-0.12034347,0.04974607,0.8200,0.1357534,1.001322,832.8691,1232.9676,NA,NA,NA,NA,NA,NA,NA,NA
Adaptive.NK.cells#,(Intercept),NA,3.9619123,4.10366429,4.24409258,0.0000,0.0000000,1.002951,152.0386,333.3925,-3.909613,-3.729136,-3.555706,0,0,1.003146,759.90878,1075.2837
Adaptive.NK.cells#,Pregnancy.stageFirst trimester,Pregnancy.stage,-0.6442304,-0.47569889,-0.31211339,0.0010,0.0001000,1.002305,1750.3330,1806.0190,NA,NA,NA,NA,NA,NA,NA,NA


### sccomp boxplot prop + FDR

In [15]:
group_col= 'Pregnancy.stage'
group_col_alias = 'Pregnancy.stage'
color_pal <- c('#D2EBC8','#7DBFA7','#EE934E','#AECDE1','#8FA4AE','#BBDD78','#8AD9DF','#EBCC96','#9FD2A8','#4169E1')
levels_group = c('Unavailable','First trimester','Second trimester','Third trimester')
baseline_group = 'Unavailable'
sig_gsub = '%_in_PBMC'

In [16]:
sig_data = sccomp_result %>% 
    pivot_longer(c(contains("c_"), 
    contains("v_")), names_pattern = "([cv])_([a-zA-Z0-9]+)", 
    names_to = c("which", "stats_name"), values_to = "stats_value"
                ) %>% 
    filter(stats_name == "FDR",stats_value < 1,factor == factor
          ) %>% mutate(group2 = baseline_group
                      ) %>%  mutate(group1 = gsub(group_col_alias, '', parameter))

In [17]:
data_proportion = attr(sccomp_result,'count_data') %>% 
    select(ClinicalID,groupby,groupby_count,all_of(group_col_alias))  %>% 
    with_groups(
    .groups = ClinicalID,
    .f = ~ mutate(
      .x,
      proportion = groupby_count / sum(groupby_count, na.rm = TRUE) *100
    )
  )
for (cell_type in cell_types){
    
    plot_data <- data_proportion  %>% select(-groupby_count) %>% 
        filter(groupby == cell_type) %>% 
        rename(plot_group = !!group_col)
    baseline_mean <- plot_data %>% 
          filter(plot_group == baseline_group) %>% 
          summarise(me = mean(proportion, na.rm = TRUE)) %>% 
          pull(me)
    p_values <- sig_data %>% 
          filter(factor ==group_col_alias,
                 groupby == cell_type) %>%
          add_significance(
            p.col = 'stats_value', 
            output.col = 'stars',
            cutpoints = c(0, 0.001, 0.01, 0.05, 1),
            symbols = c("***", "**", "*", "ns")
          ) %>% 
          filter(stars != 'ns')

    p <- ggplot(plot_data, aes(x = plot_group, y = proportion)) +
      geom_boxplot(aes(color = plot_group,fill = plot_group), width = 0.5, outlier.shape = NA, color = 'black') +
      geom_jitter(aes(color = plot_group,fill = plot_group), shape = 16, width = 0.05, size = 1.5, alpha = 0.5) +#, color = 'LightGrey',stroke = 0.1
      theme_cowplot() +
      theme(
        axis.text = element_text(size = 10, colour = 'black'),
        axis.title = element_text(size = 10, colour = 'black'),
        legend.position = 'none'
      ) +
      labs(
        title = paste0(baseline_group, " as baseline (effect >0.2 in 95% CI)"),
        subtitle = '***:<0.001, **:<0.01, *:<0.05',
        y = 'Percentage', x = group_col
      ) +
      geom_hline(yintercept = as.numeric(baseline_mean), linetype = 'dashed', alpha = 0.5) +
      stat_pvalue_manual(
        p_values, 
        group1 = "group1",
        group2 = "group2",
        y = max(plot_data$proportion, na.rm = TRUE) * 1.1,
        step.increase = 0.05,
        label = "{stars}",
        size = 5, 
        hide.ns = TRUE,
        tip.length = 0.01
      )+
      scale_color_manual(values = color_pal,aesthetics = c("colour", "fill"))
    
    # 保存图表（文件名适配细胞类型和分组）
    cell_type_B = gsub('[#]', sig_gsub, cell_type)
    ggsave2(paste0(path,
                   gsub('[.|%|/]','',cell_type_B),'_',group_col,".pdf"),
            p,width=8,heigh=8)
    }

Warning message in (function (mapping = NULL, data = NULL, stat = "identity", position = "nudge", :
“Ignoring unknown parameters: `group1` and `group2`”
Warning message in (function (mapping = NULL, data = NULL, stat = "identity", position = "nudge", :
“Ignoring unknown parameters: `group1` and `group2`”
Warning message in (function (mapping = NULL, data = NULL, stat = "identity", position = "nudge", :
“Ignoring unknown parameters: `group1` and `group2`”
Warning message in (function (mapping = NULL, data = NULL, stat = "identity", position = "nudge", :
“Ignoring unknown parameters: `group1` and `group2`”
Warning message in (function (mapping = NULL, data = NULL, stat = "identity", position = "nudge", :
“Ignoring unknown parameters: `group1` and `group2`”
Warning message in (function (mapping = NULL, data = NULL, stat = "identity", position = "nudge", :
“Ignoring unknown parameters: `group1` and `group2`”
Warning message in (function (mapping = NULL, data = NULL, stat = "identity", posi

### cell x intervals to vis the effect size and FDR
#参考组可视化effect没有意义

In [18]:
cell_types_clean <- cell_types %>%
  gsub('[#]', '', .) %>%
  gsub('[.]', ' ', .)

In [19]:
data  = sccomp_result %>% 
    select(groupby,parameter,c_effect,c_FDR) %>% 
    add_significance(p.col='c_FDR',
                     output.col='stars',
                     cutpoints = c(0, 0.001, 0.01, 0.05, 1),
                     symbols = c("***", "**", "*", ""))  %>% 
    filter(!parameter %in% c('(Intercept)')) %>% 
    filter(!grepl("___D", parameter)) %>% 
    mutate(parameter = gsub(group_col, '', parameter),
           parameter = factor(parameter, levels =levels_group),
           groupby = gsub('[#]', '', groupby),
           groupby = gsub('[.]', ' ', groupby),
           groupby = factor(groupby, levels = rev(cell_types_clean))
          )

In [20]:
#showd be 79 cell type, 排除platelets、中性粒和Bas, 但包含mast cell
length(table(data$groupby))

[1] 88

In [21]:
p <- ggplot(data, aes(x = parameter, y = groupby, fill = c_effect)) +
    geom_tile(color = "white", linewidth = 0.5) +
    geom_text(aes(label = stars), color = "black", size = 3) +
    scale_fill_gradient2(low = "#008B8B", mid = "white", high = "#A52A2A", midpoint = 0, name = "Effect size") +
    theme_classic() +
    theme(
      axis.text.x = element_text(angle = 45, hjust = 1, size = 10),
      axis.text.y = element_text(size = 10),
      axis.title = element_text(size = 12, face = "bold"),
      plot.title = element_text(size = 14, face = "bold", hjust = 0.5),
      legend.title = element_text(size = 10),
      legend.key.size = unit(0.8, "cm")
    ) +
    labs(
      title = "Robust differential composition (2-9 as baseline)",
      subtitle = '***:<0.001, **:<0.01, *:<0.05',
      x = "Age group", y = "Cell type"
    )

ggsave2(paste0(path,'Effect_size_',classification,".pdf"),p,width=8,heigh=16)

Warning message in grid.Call.graphics(C_text, as.graphicsAnnot(x$label), x$x, x$y, :
“conversion failure on 'Proliferative Vδ2+ T cells' in 'mbcsToSbcs': for δ (U+03B4)”
Warning message in grid.Call.graphics(C_text, as.graphicsAnnot(x$label), x$x, x$y, :
“conversion failure on 'CD62Lhi GZMK+ Vδ2+ T cells' in 'mbcsToSbcs': for δ (U+03B4)”
Warning message in grid.Call.graphics(C_text, as.graphicsAnnot(x$label), x$x, x$y, :
“conversion failure on 'GZMB+ Vδ2+ T cells' in 'mbcsToSbcs': for δ (U+03B4)”
Warning message in grid.Call.graphics(C_text, as.graphicsAnnot(x$label), x$x, x$y, :
“conversion failure on 'GZMK+ Vδ2+ T cells' in 'mbcsToSbcs': for δ (U+03B4)”
Warning message in grid.Call.graphics(C_text, as.graphicsAnnot(x$label), x$x, x$y, :
“conversion failure on 'GZMK+ effector Vδ1+ T cells' in 'mbcsToSbcs': for δ (U+03B4)”
Warning message in grid.Call.graphics(C_text, as.graphicsAnnot(x$label), x$x, x$y, :
“conversion failure on 'KLRC2+ effector Vδ1+ T cells' in 'mbcsToSbcs': for δ (U+

## Classification_L3

### Data loading for sccomp

In [136]:
group = 'Pregnancy.stage'
classification = 'Classification_L3_PBMC_pregnancy'
#output
path = paste0('/home/liyanguo/MyImmuCell/08_supp_outputs/sccomp_age/',classification,'/')
system(paste0("mkdir -p ",path))
system(paste0("rm ",path,'*'))

In [137]:
Level_counts_frac = read.xlsx("/home/liyanguo/MyImmuCell/06_Finnal_raw_count/Classification_L3_donor_counts_frac_clinicl_table.xlsx",
                              rowNames=T)

In [138]:
Level_counts_frac

,ClinicalID,Donor_total_cell_counts,ClinicalID_PBMC_count,ClinicalID_NDN_BAS_count,ASDC#,Atypical.B.cells#,CD4+.Treg#,CD8+.Tcm#,CD8+.Tem#,CD8+.Treg#,⋯,ALP,TC,TG,HDL-C,LDL-C,Cr,UREA,UA,Glucose,BMI
,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,⋯,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
0,D0009,50065,33282,16783,6,36,419,147,5999,2,⋯,77,8.10,0.88,1.47,5.56,89.5,4.56,330.1,5.22,19.98
1,D0010,54157,15215,38942,2,29,175,39,2034,0,⋯,61,4.59,1.22,1.36,2.95,87.0,4.70,357.0,5.10,21.14
2,D0011,63656,23729,39927,4,87,448,201,3368,6,⋯,83,5.67,0.90,1.61,3.68,69.6,5.80,431.9,5.23,24.09
3,D0012,56283,13370,42913,3,26,178,173,4139,0,⋯,122,3.92,2.32,1.37,2.31,60.0,5.40,387.0,6.02,23.56
4,D0013,44586,29053,15533,2,48,458,134,9887,4,⋯,NA,4.02,1.49,1.04,2.07,58.0,3.50,273.4,4.54,23.59
5,D0014,49710,12396,37314,3,111,250,27,1097,1,⋯,58,5.09,0.38,2.18,2.69,51.6,4.90,283.0,4.34,20.45
6,D0015,64811,44834,19977,5,13,600,44,2646,2,⋯,NA,3.92,1.52,1.05,2.12,74.0,5.13,265.0,5.12,23.62
7,D0016,46441,35314,11127,10,121,708,143,2848,1,⋯,433,NA,NA,NA,NA,NA,NA,NA,NA,NA
8,D0017,57178,19142,38036,5,27,351,111,4395,0,⋯,NA,NA,NA,NA,NA,82.5,5.14,NA,4.97,21.78


In [139]:
cell_types = c('ASDC#','LAMP3+.DC#', 'cDC1#','cDC2#','pDCs#',
               'Naïve.B.cells#','Atypical.B.cells#','Memory.B.cells#','Plasma.cells#',
               'DN.T.cells#',
               'Naïve.CD4+.T.cells#',
               'Help.memory.T.cells#',
               'Cytotoxic.CD4+.T.cells#',
               'CD4+.Treg#','CD8+.Treg#',
               'Naïve.CD8+.T.cells#','CD8+.Tcm#','CD8+.Tem#',
               'MAIT#','NKT#',
               'Vδ1+.T.cells#','Vδ2+.T.cells#',
               'NK.cells#','Non-NK.ILCs#',
               
               'HSPC#','Mast.cells#',
               
               'Classical.monocytes#', 'Intermediate.monocytes#','Non-classical.Monocytes#',
               'LDNs#'
)

In [140]:
#showd be 30 cell type, 排除platelets、NDNs和Bas, 但包含mast cell
length(cell_types)

[1] 30

In [141]:
Level_counts_frac_long = Level_counts_frac %>%
    select(ClinicalID,DonorID, Batch, Gender, Age_in_Years,
       `Ten_year_intervals`, Pregnancy, Pregnancy.stage, Sampling.time, all_of(cell_types)) %>%
    pivot_longer(
      cols = -c(ClinicalID,DonorID, Batch, Gender, Age_in_Years, `Ten_year_intervals`, Pregnancy, Pregnancy.stage, Sampling.time),
      names_to = "groupby",
      values_to = "groupby_count"
    ) %>%
    rename(Intervals = `Ten_year_intervals`) %>%
    filter(Pregnancy.stage != 'Parturition') %>% #a single sample
    mutate(
      groupby_count = as.integer(groupby_count),
      Intervals = factor(Intervals, levels = c('2-9','10-19','20-29','30-39','40-49','50-59','60-69','70-79','80-89','90+')),
      Gender = factor(Gender, levels = c('Female','Male')),
      Pregnancy = factor(Pregnancy, levels = c('NO','YES')),
      Sampling.time = factor(Sampling.time, levels = c('Morning','Evening')),
      Pregnancy.stage = factor(Pregnancy.stage, levels = c('Unavailable','First trimester','Second trimester','Third trimester'))
    ) %>% {
        pregnant <- filter(., Pregnancy == 'YES')
        # 计算怀孕样本数量
        n_pregnant <- n_distinct(pregnant$DonorID)
        
        # 非怀孕样本
        non_pregnant <- filter(., Pregnancy == 'NO') %>% filter(., Age_in_Years >= 18 & Age_in_Years <= 40) #%>%
            #distinct(DonorID, .keep_all = TRUE) %>%  # 按样本去重
            #slice_sample(n = n_pregnant, replace = FALSE)  # 无放回抽样
        
        # 合并怀孕样本和对照样本（保留原始长格式结构）
        bind_rows(
            pregnant,
            filter(., DonorID %in% non_pregnant$DonorID)  # 匹配非怀孕样本的所有行
        )
    }

### Best model and plot

In [142]:
#年龄作为fixed effect
#性别作为fixed effect，但效果一般。已有研究表明性别对于免疫细胞比例影响局限在NK和单核细胞 (Effects of sex and aging on the immune cell landscape as assessed by single-cell transcriptomic analysis)
#仅考虑个体间DonorID的差异和实验批次Batch的差异,使用随机截距Intercept, 旨在捕捉个体和批次基线差异

In [143]:
sccomp_result = Level_counts_frac_long %>%
    sccomp_estimate(
    formula_composition = ~ Pregnancy.stage, # + (1 | DonorID)
    sample = "ClinicalID",
    cell_group = "groupby",
    abundance = "groupby_count",
    bimodal_mean_variability_association = TRUE,
    cores = 48, verbose = FALSE
    ) %>% sccomp_remove_outliers(cores = 48, verbose = FALSE, max_sampling_iterations = 2000) %>% sccomp_test(test_composition_above_logit_fold_change = 0.2)

sccomp says: groupby_count column is an integer. The sum-constrained beta binomial model will be used

sccomp says: estimation

sccomp says: the composition design matrix has columns: (Intercept), Pregnancy.stageFirst trimester, Pregnancy.stageSecond trimester, Pregnancy.stageThird trimester

sccomp says: the variability design matrix has columns: (Intercept)

Loading model from cache...

sccomp says: to do hypothesis testing run `sccomp_test()`,
  the `test_composition_above_logit_fold_change` = 0.1 equates to a change of ~10%, and
  0.7 equates to ~100% increase, if the baseline is ~0.1 proportion.
  Use `sccomp_proportional_fold_change` to convert c_effect (linear) to proportion difference (non-linear).

Loading model from cache...



Running standalone generated quantities after 1 MCMC chain, with 48 thread(s) per chain...

Chain 1 finished in 0.0 seconds.


sccomp says: outlier identification - step 1/2

Loading model from cache...



Running standalone generated quantities after 1 MCMC chain, with 48 thread(s) per chain...

Chain 1 finished in 0.0 seconds.


sccomp says: outlier-free model fitting - step 2/2

sccomp says: the composition design matrix has columns: (Intercept), Pregnancy.stageFirst trimester, Pregnancy.stageSecond trimester, Pregnancy.stageThird trimester

sccomp says: the variability design matrix has columns: (Intercept)

Loading model from cache...



In [144]:
saveRDS(sccomp_result,paste0(dirname(path),'/',classification,"sccomp_result.RDS"))

In [145]:
p1 = sccomp_boxplot(sccomp_result, factor = "Pregnancy.stage",)
ggsave2(paste0(dirname(path),'/',classification, "sccomp_boxplot.pdf"),
        p1,width=80,height=20,limitsize = FALSE)

sccomp says: When visualising proportions, especially for complex models, consider setting `remove_unwanted_effects=TRUE`. This will adjust the proportions, preserving only the observed effect.

Loading model from cache...



Running standalone generated quantities after 1 MCMC chain, with 1 thread(s) per chain...

Chain 1 finished in 0.0 seconds.


Joining with `by = join_by(groupby, ClinicalID)`
Joining with `by = join_by(groupby, Pregnancy.stage)`
Warning message in grid.Call.graphics(C_text, as.graphicsAnnot(x$label), x$x, x$y, :
“conversion failure on 'Vδ1+.T.cells#' in 'mbcsToSbcs': for δ (U+03B4)”
Warning message in grid.Call.graphics(C_text, as.graphicsAnnot(x$label), x$x, x$y, :
“conversion failure on 'Vδ2+.T.cells#' in 'mbcsToSbcs': for δ (U+03B4)”


In [146]:
p2 = plot_1D_intervals(sccomp_result[!grepl("___D", sccomp_result$parameter),])
ggsave2(paste0(dirname(path),'/',classification, "sccomp_plot_1D.pdf"),
        p2,width=20,height=20,limitsize = FALSE)

Warning message in grid.Call.graphics(C_text, as.graphicsAnnot(x$label), x$x, x$y, :
“conversion failure on 'Vδ2+.T.cells#' in 'mbcsToSbcs': for δ (U+03B4)”
Warning message in grid.Call.graphics(C_text, as.graphicsAnnot(x$label), x$x, x$y, :
“conversion failure on 'Vδ1+.T.cells#' in 'mbcsToSbcs': for δ (U+03B4)”
Warning message in grid.Call.graphics(C_text, as.graphicsAnnot(x$label), x$x, x$y, :
“conversion failure on 'Vδ1+.T.cells#' in 'mbcsToSbcs': for δ (U+03B4)”
Warning message in grid.Call.graphics(C_text, as.graphicsAnnot(x$label), x$x, x$y, :
“conversion failure on 'Vδ2+.T.cells#' in 'mbcsToSbcs': for δ (U+03B4)”
Warning message in grid.Call.graphics(C_text, as.graphicsAnnot(x$label), x$x, x$y, :
“conversion failure on 'Vδ1+.T.cells#' in 'mbcsToSbcs': for δ (U+03B4)”
Warning message in grid.Call.graphics(C_text, as.graphicsAnnot(x$label), x$x, x$y, :
“conversion failure on 'Vδ2+.T.cells#' in 'mbcsToSbcs': for δ (U+03B4)”


In [147]:
p3 = plot_2D_intervals(sccomp_result)
ggsave2(paste0(dirname(path),'/',classification, "sccomp_plot_2D.pdf"),
        p3,width=20,height=20,limitsize = FALSE)

In [148]:
head(sccomp_result)

groupby,parameter,factor,c_lower,c_effect,c_upper,c_pH0,c_FDR,c_rhat,c_ess_bulk,c_ess_tail,v_lower,v_effect,v_upper,v_pH0,v_FDR,v_rhat,v_ess_bulk,v_ess_tail
<chr>,<chr>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
ASDC#,(Intercept),NA,-3.47276073,-3.284818695,-3.1213959,0.0000,0.0000000,1.0029537,77.53721,194.4944,-11.02920,-10.468138,-9.997665,0,0,1.002232,269.8871,345.4783
ASDC#,Pregnancy.stageFirst trimester,Pregnancy.stage,-0.25675831,-0.066318579,0.1210086,0.9220,0.5216458,0.9999692,769.81912,1668.2877,NA,NA,NA,NA,NA,NA,NA,NA
ASDC#,Pregnancy.stageSecond trimester,Pregnancy.stage,-0.07305722,0.152859808,0.3672115,0.6735,0.1950526,1.0036757,454.32356,792.4391,NA,NA,NA,NA,NA,NA,NA,NA
ASDC#,Pregnancy.stageThird trimester,Pregnancy.stage,-0.26125399,-0.003155425,0.2209856,0.9395,0.3266429,1.0015316,268.88317,339.6486,NA,NA,NA,NA,NA,NA,NA,NA
Atypical.B.cells#,(Intercept),NA,-1.11592972,-0.944790876,-0.7882098,0.0000,0.0000000,1.0004055,75.17430,118.2631,-8.09552,-7.854244,-7.620044,0,0,1.000123,1852.7052,1816.9746
Atypical.B.cells#,Pregnancy.stageFirst trimester,Pregnancy.stage,-0.09528481,0.094357118,0.2765725,0.8720,0.3934722,0.9998527,1858.82197,1662.5503,NA,NA,NA,NA,NA,NA,NA,NA


### sccomp boxplot prop + intervals + FDR

In [149]:
group_col= 'Pregnancy.stage'
group_col_alias = 'Pregnancy.stage'
color_pal <- c('#D2EBC8','#7DBFA7','#EE934E','#AECDE1','#8FA4AE','#BBDD78','#8AD9DF','#EBCC96','#9FD2A8','#4169E1')
levels_group = c('Unavailable','First trimester','Second trimester','Third trimester')
baseline_group = 'Unavailable'
sig_gsub = '%_in_PBMC'

In [150]:
sig_data = sccomp_result %>% 
    pivot_longer(c(contains("c_"),
    contains("v_")), names_pattern = "([cv])_([a-zA-Z0-9]+)", 
    names_to = c("which", "stats_name"), values_to = "stats_value"
                ) %>% 
    filter(stats_name == "FDR",stats_value < 1,factor == factor
          ) %>% mutate(group2 = baseline_group
                      ) %>%  mutate(group1 = gsub(group_col_alias, '', parameter))

In [158]:
data_proportion = attr(sccomp_result,'count_data') %>% 
    select(ClinicalID,groupby,groupby_count,all_of(group_col_alias))  %>% 
    with_groups(
    .groups = ClinicalID,
    .f = ~ mutate(
      .x,
      proportion = groupby_count / sum(groupby_count, na.rm = TRUE) *100
    )
  )
for (cell_type in cell_types){
    
    plot_data <- data_proportion  %>% select(-groupby_count) %>% 
        filter(groupby == cell_type) %>% 
        rename(plot_group = !!group_col)
    baseline_mean <- plot_data %>% 
          filter(plot_group == baseline_group) %>% 
          summarise(me = mean(proportion, na.rm = TRUE)) %>% 
          pull(me)
    p_values <- sig_data %>% 
          filter(factor ==group_col_alias,
                 groupby == cell_type) %>%
          add_significance(
            p.col = 'stats_value', 
            output.col = 'stars',
            cutpoints = c(0, 0.001, 0.01, 0.05, 1),
            symbols = c("***", "**", "*", "ns")
          ) %>% 
          filter(stars != 'ns')

    p <- ggplot(plot_data, aes(x = plot_group, y = proportion)) +
      geom_boxplot(aes(color = plot_group,fill = plot_group), width = 0.5, outlier.shape = NA, color = 'black') +
      geom_jitter(aes(color = plot_group,fill = plot_group), shape = 16, width = 0.05, size = 1.5, alpha = 0.5) +#, color = 'LightGrey',stroke = 0.1
      theme_cowplot() +
      theme(
        axis.text = element_text(size = 10, colour = 'black'),
        axis.title = element_text(size = 10, colour = 'black'),
        legend.position = 'none'
      ) +
      labs(
        title = paste0(baseline_group, " as baseline (effect >0.2 in 95% CI)"),
        subtitle = '***:<0.001, **:<0.01, *:<0.05',
        y = 'Percentage', x = group_col
      ) +
      geom_hline(yintercept = as.numeric(baseline_mean), linetype = 'dashed', alpha = 0.5) +
      stat_pvalue_manual(
        p_values, 
        group1 = "group1",
        group2 = "group2",
        y = max(plot_data$proportion, na.rm = TRUE) * 1.1,
        step.increase = 0.05,
        label = "{stars}",
        size = 5, 
        hide.ns = TRUE,
        tip.length = 0.01
      )+
      scale_color_manual(values = color_pal,aesthetics = c("colour", "fill"))
    
    # 保存图表（文件名适配细胞类型和分组）
    cell_type_B = gsub('[#]', sig_gsub, cell_type)
    ggsave2(paste0(path,
                   gsub('[.|%|/]','',cell_type_B),'_',group_col,".pdf"),
            p,width=8,heigh=8)
    }

Warning message in (function (mapping = NULL, data = NULL, stat = "identity", position = "nudge", :
“Ignoring unknown parameters: `group1` and `group2`”
Warning message in (function (mapping = NULL, data = NULL, stat = "identity", position = "nudge", :
“Ignoring unknown parameters: `group1` and `group2`”
Warning message in (function (mapping = NULL, data = NULL, stat = "identity", position = "nudge", :
“Ignoring unknown parameters: `group1` and `group2`”
Warning message in (function (mapping = NULL, data = NULL, stat = "identity", position = "nudge", :
“Ignoring unknown parameters: `group1` and `group2`”
Warning message in (function (mapping = NULL, data = NULL, stat = "identity", position = "nudge", :
“Ignoring unknown parameters: `group1` and `group2`”
Warning message in (function (mapping = NULL, data = NULL, stat = "identity", position = "nudge", :
“Ignoring unknown parameters: `group1` and `group2`”
Warning message in (function (mapping = NULL, data = NULL, stat = "identity", posi

### cell x intervals to vis the effect size and FDR
#参考组可视化effect没有意义

In [152]:
cell_types_clean <- cell_types %>%
  gsub('[#]', '', .) %>%
  gsub('[.]', ' ', .)

In [153]:
data  = sccomp_result %>% 
    select(groupby,parameter,c_effect,c_FDR) %>% 
    add_significance(p.col='c_FDR',
                     output.col='stars',
                     cutpoints = c(0, 0.001, 0.01, 0.05, 1),
                     symbols = c("***", "**", "*", ""))  %>% 
    filter(!parameter %in% c('(Intercept)')) %>% 
    filter(!grepl("___D", parameter)) %>% 
    mutate(parameter = gsub(group_col, '', parameter),
           parameter = factor(parameter, levels =levels_group),
           groupby = gsub('[#]', '', groupby),
           groupby = gsub('[.]', ' ', groupby),
           groupby = factor(groupby, levels = rev(cell_types_clean))
          )

In [154]:
#showd be 29 cell type, 排除platelets、中性粒和Bas, 但包含mast cell
length(table(data$groupby))

[1] 30

In [155]:
p <- ggplot(data, aes(x = parameter, y = groupby, fill = c_effect)) +
    geom_tile(color = "white", linewidth = 0.5) +
    geom_text(aes(label = stars), color = "black", size = 3) +
    scale_fill_gradient2(low = "#008B8B", mid = "white", high = "#A52A2A", midpoint = 0, name = "Effect size") +
    theme_classic() +
    theme(
      axis.text.x = element_text(angle = 45, hjust = 1, size = 10),
      axis.text.y = element_text(size = 10),
      axis.title = element_text(size = 12, face = "bold"),
      plot.title = element_text(size = 14, face = "bold", hjust = 0.5),
      legend.title = element_text(size = 10),
      legend.key.size = unit(0.8, "cm")
    ) +
    labs(
      title = "Robust differential composition (2-9 as baseline)",
      subtitle = '***:<0.001, **:<0.01, *:<0.05',
      x = "Age group", y = "Cell type"
    )

ggsave2(paste0(path,'Effect_size_',classification,".pdf"),p,width=8,heigh=16)

Warning message in grid.Call.graphics(C_text, as.graphicsAnnot(x$label), x$x, x$y, :
“conversion failure on 'Vδ2+ T cells' in 'mbcsToSbcs': for δ (U+03B4)”
Warning message in grid.Call.graphics(C_text, as.graphicsAnnot(x$label), x$x, x$y, :
“conversion failure on 'Vδ1+ T cells' in 'mbcsToSbcs': for δ (U+03B4)”


cell x  gender/pregnancy/circadian rhythm to vis the effect size and FDR
#参考组的截距可视化effect没有意义

# NDNs

#Exclude PBMC+platelet, basophils also not mention

## Classification_L4

### Data loading for sccomp

In [42]:
group = 'Pregnancy.stage'
classification = 'Classification_L4_NDNs_pregnancy'
#output
path = paste0('/home/liyanguo/MyImmuCell/08_supp_outputs/sccomp_age/',classification,'/')
system(paste0("mkdir -p ",path))
system(paste0("rm ",path,'*'))

In [43]:
Level_counts_frac = read.xlsx("/home/liyanguo/MyImmuCell/06_Finnal_raw_count/Classification_L4_donor_counts_frac_clinicl_table.xlsx",
                              rowNames=T)

In [44]:
cell_types = c('Core.NDNs#','IRF1-.GBP2-.NDNs#','IFIT2-.RNF213-.NDNs#',
               'VIM-.FLNA-.NDNs#',
                'CXCL8-.PTGS2+.NDNs#',
                'FOS-.NDNs#', 'MT-ATP6-.MT-CO2-.NDNs#',
               'ALPL-.MARCKS-.NDNs#','RGS2-.NDNs#'
)

In [45]:
#showd be 6 cell type, 排除platelets、中性粒和Bas, 但包含mast cell
length(cell_types)

[1] 9

In [46]:
Level_counts_frac_long = Level_counts_frac %>%
    select(ClinicalID,DonorID, Batch, Gender, Age_in_Years,
       `Ten_year_intervals`, Pregnancy, Pregnancy.stage, Sampling.time, all_of(cell_types)) %>%
    pivot_longer(
      cols = -c(ClinicalID,DonorID, Batch, Gender, Age_in_Years, `Ten_year_intervals`, Pregnancy, Pregnancy.stage, Sampling.time),
      names_to = "groupby",
      values_to = "groupby_count"
    ) %>%
    rename(Intervals = `Ten_year_intervals`) %>%
    filter(Pregnancy.stage != 'Parturition') %>% #a single sample
    mutate(
      groupby_count = as.integer(groupby_count),
      Intervals = factor(Intervals, levels = c('2-9','10-19','20-29','30-39','40-49','50-59','60-69','70-79','80-89','90+')),
      Gender = factor(Gender, levels = c('Female','Male')),
      Pregnancy = factor(Pregnancy, levels = c('NO','YES')),
      Sampling.time = factor(Sampling.time, levels = c('Morning','Evening')),
      Pregnancy.stage = factor(Pregnancy.stage, levels = c('Unavailable','First trimester','Second trimester','Third trimester'))
    ) %>% {
        pregnant <- filter(., Pregnancy == 'YES')
        # 计算怀孕样本数量
        n_pregnant <- n_distinct(pregnant$DonorID)
        
        # 非怀孕样本
        non_pregnant <- filter(., Pregnancy == 'NO') %>% filter(., Age_in_Years >= 18 & Age_in_Years <= 40) #%>%
            #distinct(DonorID, .keep_all = TRUE) %>%  # 按样本去重
            #slice_sample(n = n_pregnant, replace = FALSE)  # 无放回抽样
        
        # 合并怀孕样本和对照样本（保留原始长格式结构）
        bind_rows(
            pregnant,
            filter(., DonorID %in% non_pregnant$DonorID)  # 匹配非怀孕样本的所有行
        )
    }

In [47]:
head(Level_counts_frac_long)

ClinicalID,DonorID,Batch,Gender,Age_in_Years,Intervals,Pregnancy,Pregnancy.stage,Sampling.time,groupby,groupby_count
<chr>,<chr>,<chr>,<fct>,<dbl>,<fct>,<fct>,<fct>,<fct>,<chr>,<int>
D0052_FT,D0052,B35,Female,37,30-39,YES,First trimester,Morning,Core.NDNs#,26678
D0052_FT,D0052,B35,Female,37,30-39,YES,First trimester,Morning,IRF1-.GBP2-.NDNs#,1462
D0052_FT,D0052,B35,Female,37,30-39,YES,First trimester,Morning,IFIT2-.RNF213-.NDNs#,2589
D0052_FT,D0052,B35,Female,37,30-39,YES,First trimester,Morning,VIM-.FLNA-.NDNs#,3546
D0052_FT,D0052,B35,Female,37,30-39,YES,First trimester,Morning,CXCL8-.PTGS2+.NDNs#,1393
D0052_FT,D0052,B35,Female,37,30-39,YES,First trimester,Morning,FOS-.NDNs#,3250


### Best model and plot

In [48]:
#年龄作为fixed effect
#性别作为fixed effect，但效果一般。已有研究表明性别对于免疫细胞比例影响局限在NK和单核细胞 (Effects of sex and aging on the immune cell landscape as assessed by single-cell transcriptomic analysis)
#仅考虑个体间DonorID的差异和实验批次Batch的差异,使用随机截距Intercept, 旨在捕捉个体和批次基线差异

In [49]:
sccomp_result = Level_counts_frac_long %>%
    sccomp_estimate(
    formula_composition = ~ Pregnancy.stage,
    sample = "ClinicalID",
    cell_group = "groupby",
    abundance = "groupby_count",
    bimodal_mean_variability_association = TRUE,
    cores = 48, verbose = FALSE
    ) %>% sccomp_remove_outliers(cores = 48, verbose = FALSE, max_sampling_iterations = 2000) %>% sccomp_test(test_composition_above_logit_fold_change = 0.2)

sccomp says: groupby_count column is an integer. The sum-constrained beta binomial model will be used

sccomp says: estimation

sccomp says: the composition design matrix has columns: (Intercept), Pregnancy.stageFirst trimester, Pregnancy.stageSecond trimester, Pregnancy.stageThird trimester

sccomp says: the variability design matrix has columns: (Intercept)

Loading model from cache...

sccomp says: to do hypothesis testing run `sccomp_test()`,
  the `test_composition_above_logit_fold_change` = 0.1 equates to a change of ~10%, and
  0.7 equates to ~100% increase, if the baseline is ~0.1 proportion.
  Use `sccomp_proportional_fold_change` to convert c_effect (linear) to proportion difference (non-linear).

Loading model from cache...



Running standalone generated quantities after 1 MCMC chain, with 48 thread(s) per chain...

Chain 1 finished in 0.0 seconds.


sccomp says: outlier identification - step 1/2

Loading model from cache...



Running standalone generated quantities after 1 MCMC chain, with 48 thread(s) per chain...

Chain 1 finished in 0.0 seconds.


sccomp says: outlier-free model fitting - step 2/2

sccomp says: the composition design matrix has columns: (Intercept), Pregnancy.stageFirst trimester, Pregnancy.stageSecond trimester, Pregnancy.stageThird trimester

sccomp says: the variability design matrix has columns: (Intercept)

Loading model from cache...



In [50]:
saveRDS(sccomp_result,paste0(dirname(path),'/',classification,"sccomp_result.RDS"))

In [51]:
p1 = sccomp_boxplot(sccomp_result, factor = "Pregnancy.stage",)
ggsave2(paste0(dirname(path),'/',classification, "sccomp_boxplot.pdf"),
        p1,width=80,height=20,limitsize = FALSE)

sccomp says: When visualising proportions, especially for complex models, consider setting `remove_unwanted_effects=TRUE`. This will adjust the proportions, preserving only the observed effect.

Loading model from cache...



Running standalone generated quantities after 1 MCMC chain, with 1 thread(s) per chain...

Chain 1 finished in 0.0 seconds.


Joining with `by = join_by(groupby, ClinicalID)`
Joining with `by = join_by(groupby, Pregnancy.stage)`


In [52]:
p2 = plot_1D_intervals(sccomp_result[!grepl("___D", sccomp_result$parameter),])
ggsave2(paste0(dirname(path),'/',classification, "sccomp_plot_1D.pdf"),
        p2,width=20,height=20,limitsize = FALSE)

In [53]:
p3 = plot_2D_intervals(sccomp_result)
ggsave2(paste0(dirname(path),'/',classification, "sccomp_plot_2D.pdf"),
        p3,width=20,height=20,limitsize = FALSE)

In [54]:
head(sccomp_result)

groupby,parameter,factor,c_lower,c_effect,c_upper,c_pH0,c_FDR,c_rhat,c_ess_bulk,c_ess_tail,v_lower,v_effect,v_upper,v_pH0,v_FDR,v_rhat,v_ess_bulk,v_ess_tail
<chr>,<chr>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
ALPL-.MARCKS-.NDNs#,(Intercept),NA,-0.1702863,-0.1278737,-0.08473491,0.9990,0.2553571,0.9998729,2021.367,2004.139,-5.263563,-5.056510,-4.848686,0,0,1.002383,2036.545,1847.15
ALPL-.MARCKS-.NDNs#,Pregnancy.stageFirst trimester,Pregnancy.stage,-0.3943443,-0.2434793,-0.10061310,0.2690,0.0540000,1.0010566,1898.305,1746.320,NA,NA,NA,NA,NA,NA,NA,NA
ALPL-.MARCKS-.NDNs#,Pregnancy.stageSecond trimester,Pregnancy.stage,-0.6547757,-0.4803048,-0.31890939,0.0005,0.0002500,1.0045015,1962.260,1712.900,NA,NA,NA,NA,NA,NA,NA,NA
ALPL-.MARCKS-.NDNs#,Pregnancy.stageThird trimester,Pregnancy.stage,-0.5996832,-0.3653736,-0.14300611,0.0675,0.0277000,0.9995350,1887.398,1917.345,NA,NA,NA,NA,NA,NA,NA,NA
CXCL8-.PTGS2+.NDNs#,(Intercept),NA,-0.2683826,-0.2183520,-0.16898946,0.2235,0.0447000,0.9995729,2019.686,1588.407,-4.921295,-4.719441,-4.520683,0,0,1.000356,1796.137,1744.04
CXCL8-.PTGS2+.NDNs#,Pregnancy.stageFirst trimester,Pregnancy.stage,-0.6687028,-0.4947483,-0.32677098,0.0000,0.0000000,1.0001214,2065.112,2012.340,NA,NA,NA,NA,NA,NA,NA,NA


### sccomp boxplot prop + intervals + FDR

In [55]:
sccomp_result = readRDS(paste0(dirname(path),'/',classification,"sccomp_result.RDS"))

In [56]:
group_col= 'Pregnancy.stage'
group_col_alias = 'Pregnancy.stage'
color_pal <- c('#D2EBC8','#7DBFA7','#EE934E','#AECDE1','#8FA4AE','#BBDD78','#8AD9DF','#EBCC96','#9FD2A8','#4169E1')
levels_group = c('Unavailable','First trimester','Second trimester','Third trimester')
baseline_group = 'Unavailable'
sig_gsub = '%_in_NEU_BAS'

In [57]:
sig_data = sccomp_result %>% 
    pivot_longer(c(contains("c_"), 
    contains("v_")), names_pattern = "([cv])_([a-zA-Z0-9]+)", 
    names_to = c("which", "stats_name"), values_to = "stats_value"
                ) %>% 
    filter(stats_name == "FDR",stats_value < 1,factor == factor
          ) %>% mutate(group2 = baseline_group
                      ) %>%  mutate(group1 = gsub(group_col_alias, '', parameter))

In [58]:
data_proportion = attr(sccomp_result,'count_data') %>% 
    select(ClinicalID,groupby,groupby_count,all_of(group_col_alias))  %>% 
    with_groups(
    .groups = ClinicalID,
    .f = ~ mutate(
      .x,
      proportion = groupby_count / sum(groupby_count, na.rm = TRUE) *100
    )
  )
for (cell_type in cell_types){
    
    plot_data <- data_proportion  %>% select(-groupby_count) %>% 
        filter(groupby == cell_type) %>% 
        rename(plot_group = !!group_col)
    baseline_mean <- plot_data %>% 
          filter(plot_group == baseline_group) %>% 
          summarise(me = mean(proportion, na.rm = TRUE)) %>% 
          pull(me)
    p_values <- sig_data %>% 
          filter(factor ==group_col_alias,
                 groupby == cell_type) %>%
          add_significance(
            p.col = 'stats_value', 
            output.col = 'stars',
            cutpoints = c(0, 0.001, 0.01, 0.05, 1),
            symbols = c("***", "**", "*", "ns")
          ) %>% 
          filter(stars != 'ns')

    p <- ggplot(plot_data, aes(x = plot_group, y = proportion)) +
      geom_boxplot(aes(color = plot_group,fill = plot_group), width = 0.5, outlier.shape = NA, color = 'black') +
      geom_jitter(aes(color = plot_group,fill = plot_group), shape = 16, width = 0.05, size = 1.5, alpha = 0.5) +#, color = 'LightGrey',stroke = 0.1
      theme_cowplot() +
      theme(
        axis.text = element_text(size = 10, colour = 'black'),
        axis.title = element_text(size = 10, colour = 'black'),
        legend.position = 'none'
      ) +
      labs(
        title = paste0(baseline_group, " as baseline (effect >0.2 in 95% CI)"),
        subtitle = '***:<0.001, **:<0.01, *:<0.05',
        y = 'Percentage', x = group_col
      ) +
      geom_hline(yintercept = as.numeric(baseline_mean), linetype = 'dashed', alpha = 0.5) +
      stat_pvalue_manual(
        p_values, 
        group1 = "group1",
        group2 = "group2",
        y = max(plot_data$proportion, na.rm = TRUE) * 1.1,
        step.increase = 0.05,
        label = "{stars}",
        size = 5, 
        hide.ns = TRUE,
        tip.length = 0.01
      )+
      scale_color_manual(values = color_pal,aesthetics = c("colour", "fill"))
    
    # 保存图表（文件名适配细胞类型和分组）
    cell_type_B = gsub('[#]', sig_gsub, cell_type)
    ggsave2(paste0(path,
                   gsub('[.|%|/]','',cell_type_B),'_',group_col,".pdf"),
            p,width=8,heigh=8)
    }

Warning message in (function (mapping = NULL, data = NULL, stat = "identity", position = "nudge", :
“Ignoring unknown parameters: `group1` and `group2`”
Warning message in (function (mapping = NULL, data = NULL, stat = "identity", position = "nudge", :
“Ignoring unknown parameters: `group1` and `group2`”
Warning message in (function (mapping = NULL, data = NULL, stat = "identity", position = "nudge", :
“Ignoring unknown parameters: `group1` and `group2`”


### cell x intervals to vis the effect size and FDR
#参考组可视化effect没有意义

In [59]:
cell_types_clean <- cell_types %>%
  gsub('[#]', '', .) %>%
  gsub('[.]', ' ', .)

In [60]:
data  = sccomp_result %>% 
    select(groupby,parameter,c_effect,c_FDR) %>% 
    add_significance(p.col='c_FDR',
                     output.col='stars',
                     cutpoints = c(0, 0.001, 0.01, 0.05, 1),
                     symbols = c("***", "**", "*", ""))  %>% 
    filter(!parameter %in% c('(Intercept)')) %>% 
    filter(!grepl("___D", parameter)) %>% 
    mutate(parameter = gsub(group_col, '', parameter),
           parameter = factor(parameter, levels =levels_group),
           groupby = gsub('[#]', '', groupby),
           groupby = gsub('[.]', ' ', groupby),
           groupby = factor(groupby, levels = rev(cell_types_clean))
          )

In [61]:
#showd be 79 cell type, 排除platelets、中性粒和Bas, 但包含mast cell
length(table(data$groupby))

[1] 9

In [62]:
p <- ggplot(data, aes(x = parameter, y = groupby, fill = c_effect)) +
    geom_tile(color = "white", linewidth = 0.5) +
    geom_text(aes(label = stars), color = "black", size = 3) +
    scale_fill_gradient2(low = "#008B8B", mid = "white", high = "#A52A2A", midpoint = 0, name = "Effect size") +
    theme_classic() +
    theme(
      axis.text.x = element_text(angle = 45, hjust = 1, size = 10),
      axis.text.y = element_text(size = 10),
      axis.title = element_text(size = 12, face = "bold"),
      plot.title = element_text(size = 14, face = "bold", hjust = 0.5),
      legend.title = element_text(size = 10),
      legend.key.size = unit(0.8, "cm")
    ) +
    labs(
      title = "Robust differential composition (2-9 as baseline)",
      subtitle = '***:<0.001, **:<0.01, *:<0.05',
      x = "Age group", y = "Cell type"
    )

ggsave2(paste0(path,'Effect_size_',classification,".pdf"),p,width=8,heigh=16)

# LDN

#Exclude PBMC+platelet, basophils also not mention

## Classification_L4

### Data loading for sccomp

In [63]:
group = 'Pregnancy.stage'
classification = 'Classification_L4_LDNs_pregnancy'
#output
path = paste0('/home/liyanguo/MyImmuCell/08_supp_outputs/sccomp_age/',classification,'/')
system(paste0("mkdir -p ",path))
system(paste0("rm ",path,'*'))

In [64]:
Level_counts_frac = read.xlsx("/home/liyanguo/MyImmuCell/06_Finnal_raw_count/Classification_L4_donor_counts_frac_clinicl_table.xlsx",
                              rowNames=T)

In [65]:
colnames(Level_counts_frac)[grepl('LDN',colnames(Level_counts_frac))]

[1] "CD177int.iLDNs#"              "MMP8+.CD177+.iLDNs#"         
 [3] "MMP8+.CD177+.mLDNs#"          "MMP9+.CD177+.iLDNs#"         
 [5] "MPO+.CD177-.iLDNs#"           "MPO+.mLDNs#"                 
 [7] "MPO-.CD177-.iLDNs#"           "Proliferative.iLDNs#"        
 [9] "Proliferative.mLDNs#"         "CD177int.iLDNs%_in_PBMC"     
[11] "MMP8+.CD177+.iLDNs%_in_PBMC"  "MMP8+.CD177+.mLDNs%_in_PBMC" 
[13] "MMP9+.CD177+.iLDNs%_in_PBMC"  "MPO+.CD177-.iLDNs%_in_PBMC"  
[15] "MPO+.mLDNs%_in_PBMC"          "MPO-.CD177-.iLDNs%_in_PBMC"  
[17] "Proliferative.iLDNs%_in_PBMC" "Proliferative.mLDNs%_in_PBMC"

In [66]:
cell_types = c('MPO+.CD177-.iLDNs#','MPO-.CD177-.iLDNs#','CD177int.iLDNs#','MMP8+.CD177+.iLDNs#','MMP9+.CD177+.iLDNs#','Proliferative.iLDNs#',
               'MPO+.mLDNs#','MMP8+.CD177+.mLDNs#','Proliferative.mLDNs#'
)

In [67]:
#showd be 9 cell type, 排除platelets、中性粒和Bas, 但包含mast cell
length(cell_types)

[1] 9

In [68]:
table(Level_counts_frac$Pregnancy.stage)


 First trimester      Parturition Second trimester  Third trimester 
              18                1               12                6 
     Unavailable 
             957 

In [69]:
Level_counts_frac_long = Level_counts_frac %>%
    select(ClinicalID,DonorID, Batch, Gender, Age_in_Years,
       `Ten_year_intervals`, Pregnancy, Pregnancy.stage, Sampling.time, all_of(cell_types)) %>%
    pivot_longer(
      cols = -c(ClinicalID,DonorID, Batch, Gender, Age_in_Years, `Ten_year_intervals`, Pregnancy, Pregnancy.stage, Sampling.time),
      names_to = "groupby",
      values_to = "groupby_count"
    ) %>%
    rename(Intervals = `Ten_year_intervals`) %>%
    filter(Pregnancy.stage != 'Parturition') %>% #a single sample
    mutate(
      groupby_count = as.integer(groupby_count),
      Intervals = factor(Intervals, levels = c('2-9','10-19','20-29','30-39','40-49','50-59','60-69','70-79','80-89','90+')),
      Gender = factor(Gender, levels = c('Female','Male')),
      Pregnancy = factor(Pregnancy, levels = c('NO','YES')),
      Sampling.time = factor(Sampling.time, levels = c('Morning','Evening')),
      Pregnancy.stage = factor(Pregnancy.stage, levels = c('Unavailable','First trimester','Second trimester','Third trimester'))
    ) %>% {
        pregnant <- filter(., Pregnancy == 'YES')
        # 计算怀孕样本数量
        n_pregnant <- n_distinct(pregnant$DonorID)
        
        # 非怀孕样本
        non_pregnant <- filter(., Pregnancy == 'NO') %>% filter(., Age_in_Years >= 18 & Age_in_Years <= 40) #%>%
            #distinct(DonorID, .keep_all = TRUE) %>%  # 按样本去重
            #slice_sample(n = n_pregnant, replace = FALSE)  # 无放回抽样
        
        # 合并怀孕样本和对照样本（保留原始长格式结构）
        bind_rows(
            pregnant,
            filter(., DonorID %in% non_pregnant$DonorID)  # 匹配非怀孕样本的所有行
        )
    }

In [70]:
head(Level_counts_frac_long)

ClinicalID,DonorID,Batch,Gender,Age_in_Years,Intervals,Pregnancy,Pregnancy.stage,Sampling.time,groupby,groupby_count
<chr>,<chr>,<chr>,<fct>,<dbl>,<fct>,<fct>,<fct>,<fct>,<chr>,<int>
D0052_FT,D0052,B35,Female,37,30-39,YES,First trimester,Morning,MPO+.CD177-.iLDNs#,33
D0052_FT,D0052,B35,Female,37,30-39,YES,First trimester,Morning,MPO-.CD177-.iLDNs#,65
D0052_FT,D0052,B35,Female,37,30-39,YES,First trimester,Morning,CD177int.iLDNs#,47
D0052_FT,D0052,B35,Female,37,30-39,YES,First trimester,Morning,MMP8+.CD177+.iLDNs#,116
D0052_FT,D0052,B35,Female,37,30-39,YES,First trimester,Morning,MMP9+.CD177+.iLDNs#,82
D0052_FT,D0052,B35,Female,37,30-39,YES,First trimester,Morning,Proliferative.iLDNs#,45


### Best model and plot

In [71]:
#年龄作为fixed effect
#性别作为fixed effect，但效果一般。已有研究表明性别对于免疫细胞比例影响局限在NK和单核细胞 (Effects of sex and aging on the immune cell landscape as assessed by single-cell transcriptomic analysis)
#仅考虑个体间DonorID的差异和实验批次Batch的差异,使用随机截距Intercept, 旨在捕捉个体和批次基线差异

In [72]:
sccomp_result = Level_counts_frac_long %>%
    sccomp_estimate(
    formula_composition = ~ Pregnancy.stage,
    sample = "ClinicalID",
    cell_group = "groupby",
    abundance = "groupby_count",
    bimodal_mean_variability_association = TRUE,
    cores = 48, verbose = FALSE
    ) %>% sccomp_remove_outliers(cores = 48, verbose = FALSE, max_sampling_iterations = 2000) %>% sccomp_test(test_composition_above_logit_fold_change = 0.2)

sccomp says: groupby_count column is an integer. The sum-constrained beta binomial model will be used

sccomp says: estimation

sccomp says: the composition design matrix has columns: (Intercept), Pregnancy.stageFirst trimester, Pregnancy.stageSecond trimester, Pregnancy.stageThird trimester

sccomp says: the variability design matrix has columns: (Intercept)

Loading model from cache...

sccomp says: to do hypothesis testing run `sccomp_test()`,
  the `test_composition_above_logit_fold_change` = 0.1 equates to a change of ~10%, and
  0.7 equates to ~100% increase, if the baseline is ~0.1 proportion.
  Use `sccomp_proportional_fold_change` to convert c_effect (linear) to proportion difference (non-linear).

Loading model from cache...



Running standalone generated quantities after 1 MCMC chain, with 48 thread(s) per chain...

Chain 1 finished in 0.0 seconds.


sccomp says: outlier identification - step 1/2

Loading model from cache...



Running standalone generated quantities after 1 MCMC chain, with 48 thread(s) per chain...

Chain 1 finished in 0.0 seconds.


sccomp says: outlier-free model fitting - step 2/2

sccomp says: the composition design matrix has columns: (Intercept), Pregnancy.stageFirst trimester, Pregnancy.stageSecond trimester, Pregnancy.stageThird trimester

sccomp says: the variability design matrix has columns: (Intercept)

Loading model from cache...



In [73]:
saveRDS(sccomp_result,paste0(dirname(path),'/',classification,"sccomp_result.RDS"))

In [74]:
p1 = sccomp_boxplot(sccomp_result, factor = "Pregnancy.stage",)
ggsave2(paste0(dirname(path),'/',classification, "sccomp_boxplot.pdf"),
        p1,width=80,height=20,limitsize = FALSE)

sccomp says: When visualising proportions, especially for complex models, consider setting `remove_unwanted_effects=TRUE`. This will adjust the proportions, preserving only the observed effect.

Loading model from cache...



Running standalone generated quantities after 1 MCMC chain, with 1 thread(s) per chain...

Chain 1 finished in 0.0 seconds.


Joining with `by = join_by(groupby, ClinicalID)`
Joining with `by = join_by(groupby, Pregnancy.stage)`
Warning message:
“Removed 9 rows containing non-finite outside the scale range
(`stat_boxplot()`).”
Warning message:
“Removed 9 rows containing missing values or values outside the scale range
(`geom_point()`).”


In [75]:
p2 = plot_1D_intervals(sccomp_result[!grepl("___D", sccomp_result$parameter),])
ggsave2(paste0(dirname(path),'/',classification, "sccomp_plot_1D.pdf"),
        p2,width=20,height=20,limitsize = FALSE)

In [76]:
p3 = plot_2D_intervals(sccomp_result)
ggsave2(paste0(dirname(path),'/',classification, "sccomp_plot_2D.pdf"),
        p3,width=20,height=20,limitsize = FALSE)

In [77]:
head(sccomp_result)

groupby,parameter,factor,c_lower,c_effect,c_upper,c_pH0,c_FDR,c_rhat,c_ess_bulk,c_ess_tail,v_lower,v_effect,v_upper,v_pH0,v_FDR,v_rhat,v_ess_bulk,v_ess_tail
<chr>,<chr>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
CD177int.iLDNs#,(Intercept),NA,0.07009045,0.14314751,0.21699257,0.9390,0.2616111,1.0010668,1931.938,1814.214,-5.230896,-4.792942,-4.345910,0,0,1.0005478,1867.221,1712.184
CD177int.iLDNs#,Pregnancy.stageFirst trimester,Pregnancy.stage,-0.15268975,0.06181179,0.28580322,0.8945,0.3062778,1.0005092,1754.271,1680.252,NA,NA,NA,NA,NA,NA,NA,NA
CD177int.iLDNs#,Pregnancy.stageSecond trimester,Pregnancy.stage,-0.29810354,-0.10169401,0.09437674,0.8300,0.3326250,0.9995735,1696.859,1960.808,NA,NA,NA,NA,NA,NA,NA,NA
CD177int.iLDNs#,Pregnancy.stageThird trimester,Pregnancy.stage,-0.37889016,-0.10476610,0.13662565,0.7445,0.4085833,0.9997787,1887.303,1852.598,NA,NA,NA,NA,NA,NA,NA,NA
MMP8+.CD177+.iLDNs#,(Intercept),NA,0.61951778,0.69232392,0.76191588,0.0000,0.0000000,1.0028222,1956.853,1585.672,-4.553594,-4.189565,-3.842226,0,0,0.9998175,1845.187,1555.816
MMP8+.CD177+.iLDNs#,Pregnancy.stageFirst trimester,Pregnancy.stage,0.05308881,0.25270023,0.44375891,0.2925,0.1216667,1.0028389,1856.538,1835.065,NA,NA,NA,NA,NA,NA,NA,NA


### sccomp boxplot prop + intervals + FDR

In [78]:
group_col= 'Pregnancy.stage'
group_col_alias = 'Pregnancy.stage'
color_pal <- c('#D2EBC8','#7DBFA7','#EE934E','#AECDE1','#8FA4AE','#BBDD78','#8AD9DF','#EBCC96','#9FD2A8','#4169E1')
levels_group = c('Unavailable','First trimester','Second trimester','Third trimester')
baseline_group = 'Unavailable'
sig_gsub = '%_in_NEU_BAS'

In [79]:
sig_data = sccomp_result %>% 
    pivot_longer(c(contains("c_"), 
    contains("v_")), names_pattern = "([cv])_([a-zA-Z0-9]+)", 
    names_to = c("which", "stats_name"), values_to = "stats_value"
                ) %>% 
    filter(stats_name == "FDR",stats_value < 1,factor == factor
          ) %>% mutate(group2 = baseline_group
                      ) %>%  mutate(group1 = gsub(group_col_alias, '', parameter))

In [80]:
data_proportion = attr(sccomp_result,'count_data') %>% 
    select(ClinicalID,groupby,groupby_count,all_of(group_col_alias))  %>% 
    with_groups(
    .groups = ClinicalID,
    .f = ~ mutate(
      .x,
      proportion = groupby_count / sum(groupby_count, na.rm = TRUE) *100
    )
  )
for (cell_type in cell_types){
    
    plot_data <- data_proportion  %>% select(-groupby_count) %>% 
        filter(groupby == cell_type) %>% 
        rename(plot_group = !!group_col)
    baseline_mean <- plot_data %>% 
          filter(plot_group == baseline_group) %>% 
          summarise(me = mean(proportion, na.rm = TRUE)) %>% 
          pull(me)
    p_values <- sig_data %>% 
          filter(factor ==group_col_alias,
                 groupby == cell_type) %>%
          add_significance(
            p.col = 'stats_value', 
            output.col = 'stars',
            cutpoints = c(0, 0.001, 0.01, 0.05, 1),
            symbols = c("***", "**", "*", "ns")
          ) %>% 
          filter(stars != 'ns')

    p <- ggplot(plot_data, aes(x = plot_group, y = proportion)) +
      geom_boxplot(aes(color = plot_group,fill = plot_group), width = 0.5, outlier.shape = NA, color = 'black') +
      geom_jitter(aes(color = plot_group,fill = plot_group), shape = 16, width = 0.05, size = 1.5, alpha = 0.5) +#, color = 'LightGrey',stroke = 0.1
      theme_cowplot() +
      theme(
        axis.text = element_text(size = 10, colour = 'black'),
        axis.title = element_text(size = 10, colour = 'black'),
        legend.position = 'none'
      ) +
      labs(
        title = paste0(baseline_group, " as baseline (effect >0.2 in 95% CI)"),
        subtitle = '***:<0.001, **:<0.01, *:<0.05',
        y = 'Percentage', x = group_col
      ) +
      geom_hline(yintercept = as.numeric(baseline_mean), linetype = 'dashed', alpha = 0.5) +
      stat_pvalue_manual(
        p_values, 
        group1 = "group1",
        group2 = "group2",
        y = max(plot_data$proportion, na.rm = TRUE) * 1.1,
        step.increase = 0.05,
        label = "{stars}",
        size = 5, 
        hide.ns = TRUE,
        tip.length = 0.01
      )+
      scale_color_manual(values = color_pal,aesthetics = c("colour", "fill"))
    
    # 保存图表（文件名适配细胞类型和分组）
    cell_type_B = gsub('[#]', sig_gsub, cell_type)
    ggsave2(paste0(path,
                   gsub('[.|%|/]','',cell_type_B),'_',group_col,".pdf"),
            p,width=8,heigh=8)
    }

Warning message:
“Removed 1 row containing non-finite outside the scale range (`stat_boxplot()`).”
Warning message:
“Removed 1 row containing missing values or values outside the scale range
(`geom_point()`).”
Warning message in (function (mapping = NULL, data = NULL, stat = "identity", position = "nudge", :
“Ignoring unknown parameters: `group1` and `group2`”
Warning message:
“Removed 1 row containing non-finite outside the scale range (`stat_boxplot()`).”
Warning message:
“Removed 1 row containing missing values or values outside the scale range
(`geom_point()`).”
Warning message in (function (mapping = NULL, data = NULL, stat = "identity", position = "nudge", :
“Ignoring unknown parameters: `group1` and `group2`”
Warning message:
“Removed 1 row containing non-finite outside the scale range (`stat_boxplot()`).”
Warning message:
“Removed 1 row containing missing values or values outside the scale range
(`geom_point()`).”
Warning message in (function (mapping = NULL, data = NULL, stat 

### cell x intervals to vis the effect size and FDR
#参考组可视化effect没有意义

In [81]:
cell_types_clean <- cell_types %>%
  gsub('[#]', '', .) %>%
  gsub('[.]', ' ', .)

In [82]:
data  = sccomp_result %>% 
    select(groupby,parameter,c_effect,c_FDR) %>% 
    add_significance(p.col='c_FDR',
                     output.col='stars',
                     cutpoints = c(0, 0.001, 0.01, 0.05, 1),
                     symbols = c("***", "**", "*", ""))  %>% 
    filter(!parameter %in% c('(Intercept)')) %>% 
    filter(!grepl("___D", parameter)) %>% 
    mutate(parameter = gsub(group_col, '', parameter),
           parameter = factor(parameter, levels =levels_group),
           groupby = gsub('[#]', '', groupby),
           groupby = gsub('[.]', ' ', groupby),
           groupby = factor(groupby, levels = rev(cell_types_clean))
          )

In [83]:
#showd be 6 cell type, 排除platelets、中性粒和Bas, 但包含mast cell
length(table(data$groupby))

[1] 9

In [84]:
p <- ggplot(data, aes(x = parameter, y = groupby, fill = c_effect)) +
    geom_tile(color = "white", linewidth = 0.5) +
    geom_text(aes(label = stars), color = "black", size = 3) +
    scale_fill_gradient2(low = "#008B8B", mid = "white", high = "#A52A2A", midpoint = 0, name = "Effect size") +
    theme_classic() +
    theme(
      axis.text.x = element_text(angle = 45, hjust = 1, size = 10),
      axis.text.y = element_text(size = 10),
      axis.title = element_text(size = 12, face = "bold"),
      plot.title = element_text(size = 14, face = "bold", hjust = 0.5),
      legend.title = element_text(size = 10),
      legend.key.size = unit(0.8, "cm")
    ) +
    labs(
      title = "Robust differential composition (2-9 as baseline)",
      subtitle = '***:<0.001, **:<0.01, *:<0.05',
      x = "Age group", y = "Cell type"
    )

ggsave2(paste0(path,'Effect_size_',classification,".pdf"),p,width=8,heigh=16)